# Module 4: Churn Labeling
**Customer Intelligence Platform**

Goal: define what 'churned' means for this dataset, create the target label, and prepare a clean, leakage-free feature set for Module 5 (churn modeling).

In [1]:
import pandas as pd 
from sklearn.model_selection import train_test_split

customer_segments = pd.read_csv('../data/processed/customer_segments.csv') 
print(customer_segments.shape)
customer_segments.head()

(5878, 15)


,Customer ID,Recency,Frequency,Monetary,Tenure,AOV,Purchase_Variability,Product_Diversity,Frequency_log,Monetary_log,AOV_log,Purchase_Variability_log,Product_Diversity_log,Cluster,Segment_Name
0,12346.0,326,12,77556.46,400,6463.038333,65.426989,27,2.564949,11.258774,8.774010,4.196103,3.332205,1,Loyal High-Value Customers
1,12347.0,2,8,5633.32,402,704.165000,19.035055,126,2.197225,8.636632,6.558432,2.997484,4.844187,1,Loyal High-Value Customers
2,12348.0,75,5,2019.40,362,403.880000,57.703264,25,1.791759,7.611051,6.003591,4.072495,3.258097,3,At-Risk Regulars
3,12349.0,19,4,4428.69,570,1107.172500,187.040994,138,1.609438,8.396085,7.010468,5.236660,4.934474,1,Loyal High-Value Customers
4,12350.0,310,1,334.40,0,334.400000,0.000000,17,0.693147,5.815324,5.815324,0.000000,2.890372,0,New / Occasional Big-Ticket Buyers


## Step 2: Define Churn Threshold

Before picking a number, look at two things: how spread out `Recency` is across customers, and how long customers typically wait between orders when they *are* still active.

In [2]:
customer_segments['Recency'].describe()

count    5878.000000
mean      201.331916
std       209.338707
min         1.000000
25%        26.000000
50%        96.000000
75%       380.000000
max       739.000000
Name: Recency, dtype: float64

In [ ]:
for p in [50, 60, 70, 75, 80, 90, 95]: 
    print(f"{p}th percentile: {customer_segments['Recency'].quantile(p / 100):.1f} days")

50th percentile: 96.0 days
60th percentile: 189.0 days
70th percentile: 311.0 days
75th percentile: 380.0 days
80th percentile: 410.0 days
90th percentile: 535.0 days
95th percentile: 625.0 days


In [4]:
# Reload and clean the raw transactions the same way as Module 2, to get invoice-level purchase gaps
raw = pd.read_csv('../data/raw/online_retail_II.csv')
raw = raw.dropna(subset=['Customer ID'])
raw = raw[~raw['Invoice'].astype(str).str.startswith('C')]
raw = raw[(raw['Quantity'] > 0) & (raw['Price'] > 0)]
raw['InvoiceDate'] = pd.to_datetime(raw['InvoiceDate'])

invoice_dates = raw.groupby(['Customer ID', 'Invoice'])['InvoiceDate'].first().reset_index()
invoice_dates = invoice_dates.sort_values(['Customer ID', 'InvoiceDate'])
invoice_dates['gap'] = invoice_dates.groupby('Customer ID')['InvoiceDate'].diff().dt.days

invoice_dates['gap'].describe()

count    31091.000000
mean        51.239426
std         75.754392
min          0.000000
25%          6.000000
50%         24.000000
75%         61.000000
max        714.000000
Name: gap, dtype: float64

**Threshold decision: 180 days.**

The median gap between a customer's own orders is only ~24 days, and even the 75th percentile of gaps is ~61 days — so an active customer rarely goes more than 2 months without buying again. **180 days is roughly 3x that upper range**, giving genuinely inactive customers a wide buffer before being labeled churned, while still catching the ones who've clearly drifted away. It also lines up with the common 6-month churn convention used in retail.

## Step 3: Create Churn Label

In [5]:
customer_segments['Churned'] = (customer_segments['Recency'] > 180).astype(int)
customer_segments[['Customer ID', 'Recency', 'Churned']].head()

,Customer ID,Recency,Churned
0,12346.0,326,1
1,12347.0,2,0
2,12348.0,75,0
3,12349.0,19,0
4,12350.0,310,1


## Step 4: Check Class Balance

In [6]:
customer_segments['Churned'].value_counts(normalize=True).round(4)*100

Churned
0    59.17
1    40.83
Name: proportion, dtype: float64

## Step 5: Analyze Churn by Segment

In [7]:
customer_segments.groupby('Segment_Name')['Churned'].mean().round(3).sort_values(ascending=False)

Segment_Name
Lost / Churned Customers              0.849
New / Occasional Big-Ticket Buyers    0.467
At-Risk Regulars                      0.319
Loyal High-Value Customers            0.049
Name: Churned, dtype: float64

## Step 6: Prepare Feature Set (Avoid Data Leakage)

**Important:** `Churned` was derived directly from `Recency` (`Recency > 180`), so `Recency` cannot be used as an input feature — the model would just trivially re-learn the 180-day cutoff instead of finding real predictive patterns. `Customer ID` and `Segment_Name` are also excluded: ID is not a behavioral signal, and `Segment_Name` is just a text label for `Cluster` (which we keep as the numeric version).

In [8]:
feature_cols = ['Frequency_log', 'Monetary_log', 'Tenure', 'AOV_log',
                'Purchase_Variability_log', 'Product_Diversity_log', 'Cluster']
target_col = 'Churned'

customer_segments[feature_cols + [target_col]].head()

,Frequency_log,Monetary_log,Tenure,AOV_log,Purchase_Variability_log,Product_Diversity_log,Cluster,Churned
0,2.564949,11.258774,400,8.774010,4.196103,3.332205,1,1
1,2.197225,8.636632,402,6.558432,2.997484,4.844187,1,0
2,1.791759,7.611051,362,6.003591,4.072495,3.258097,3,0
3,1.609438,8.396085,570,7.010468,5.236660,4.934474,1,0
4,0.693147,5.815324,0,5.815324,0.000000,2.890372,0,1


## Step 7: Train-Test Split

In [9]:
train_df, test_df = train_test_split(
    customer_segments,
    test_size=0.2,
    random_state=42,
    stratify=customer_segments[target_col])

print('Train shape:', train_df.shape, '| Churn rate:', train_df[target_col].mean().round(4))
print('Test shape:', test_df.shape, '| Churn rate:', test_df[target_col].mean().round(4))

Train shape: (4702, 16) | Churn rate: 0.4083
Test shape: (1176, 16) | Churn rate: 0.4082


## Step 8: Save Prepared Dataset

In [10]:
train_df.to_csv('../data/processed/churn_train.csv', index=False)
test_df.to_csv('../data/processed/churn_test.csv', index=False)

print('Saved churn_train.csv and churn_test.csv to data/processed/')

Saved churn_train.csv and churn_test.csv to data/processed/


### Module 4 Summary

- **Churn threshold:** `Recency > 180 days`, justified by the typical inter-purchase gap (median ~24 days, 75th percentile ~61 days) and cross-validated against the Module 3 segments (Lost/Churned ~85% churned, Loyal High-Value ~5% churned)
- **Class balance:** ~41% churned / ~59% not churned — mild imbalance, `class_weight='balanced'` recommended for Module 5
- **Data leakage avoided:** `Recency`, `Customer ID`, and `Segment_Name` excluded from the feature set
- **Final feature set (7 features):** `Frequency_log`, `Monetary_log`, `Tenure`, `AOV_log`, `Purchase_Variability_log`, `Product_Diversity_log`, `Cluster`
- **Train/test split:** 80/20, stratified by `Churned`, saved as `churn_train.csv` / `churn_test.csv` to `data/processed/` — ready for Module 5 (Logistic Regression + Random Forest)